# 007 - JupyterLab Sidebar Chatbot 데모

JupyterLab **우측 사이드바**에 뜨는 챗봇입니다. 두뇌는 **langgraph 그래프(deepagents) + OpenAI 호환 모델**(실 OpenAI / 사내 vLLM / 로컬 Ollama) 이고, 설치부터 실행까지 **전부 노트북 셀 안에서** 끝납니다(jupyter 서버 재시작 불필요).

| 단계 | 하는 일 | jupyter 재시작? |
|---|---|---|
| ① 프론트 설치 | 셀에서 `%pip install <wheel>` → 브라우저 새로고침 → 우측 💬 탭 등장 | ❌ 불필요 |
| ② 두뇌 시작 | 셀에서 `start_graph_server(...)` (langgraph 그래프, localhost:8765) | ❌ 불필요 |
| ③ 대화 | 💬 탭에서 입력 → deepagents 응답 | — |

> ⚠️ **온라인/개발 전용**: 두뇌가 OpenAI 호환 REST 를 호출하고 `langchain-openai`/`openai`(httpx) 를 씁니다. 실 OpenAI 면 외부망, 사내 vLLM 이면 내부망입니다. 키는 인자(`api_key=...`) 또는 환경변수(`OPENAI_API_KEY`) 로만 — 노트북에 키 하드코딩 금지.
>
> ⚠️ 전제: 브라우저의 `127.0.0.1` 과 이 커널이 **같은 기계**여야 합니다(로컬/컨테이너/SSH 포트포워딩). single-file `.py` 가 아니라 빌드가 필요한 익스텐션입니다 — 자세한 건 `README.md` 참고.

## ① 프론트엔드 설치 — 노트북 셀에서 (jupyter 재시작 불필요)

터미널을 못 여는 환경이라면 **셀에서 직접 `%pip install`** 하면 됩니다. `%pip` 매직은 지금 이 커널(=jupyter 서버와 같은 env)에 설치하므로, labextension 자산이 `share/jupyter/labextensions/` 에 들어갑니다. 폐쇄망에서는 반입한 `.whl` 경로를 지정하세요.

In [ ]:
# 빌드 산출물 dist/ 의 wheel 을 설치 (`pip wheel . -w dist` 결과). 폐쇄망에선 반입한 .whl 경로로 바꾸세요.
%pip install dist/jlab_sidebar_chatbot-0.1.0-py3-none-any.whl
# 설치 후 → 브라우저에서 JupyterLab 페이지를 '새로고침'(Cmd/Ctrl+R) 하면 우측에 💬 탭이 생깁니다.
# (서버 재시작 불필요: jupyter 는 페이지를 그릴 때마다 labextensions 폴더를 다시 스캔하기 때문)

## ② 두뇌 서버 시작 — 이 셀 한 줄

커널 안에서 localhost HTTP 서버(기본 포트 8765)를 백그라운드로 띄웁니다. 우측 💬 탭에서 보낸 메시지가 이 서버로 전달됩니다. (방금 `%pip install` 한 패키지는 이 커널에서 바로 import 됩니다 — 커널 재시작도 불필요.)

In [ ]:
# 두뇌 = langgraph 그래프(deepagents + InMemorySaver). 두 가지 방식 중 택1.
from jlab_sidebar_chatbot import start_graph_server

# (a) 환경변수 방식 — OPENAI_API_KEY/_BASE_URL/_MODEL 이 jupyter env 에서 상속됨 (권장)
start_graph_server()

# (b) 사내 vLLM 한 줄로 — 키 노출이 싫으면 getpass.getpass() 로 받기
# import getpass
# start_graph_server(
#     api_key=getpass.getpass("vLLM API KEY: "),
#     base_url="https://<사내 vllm host>/v1",
#     model="<vLLM 의 served-model-name>",
# )

# → http://127.0.0.1:8765 (백그라운드 스레드). 이제 우측 💬 탭에서 대화.
# 중지: from jlab_sidebar_chatbot import stop_graph_server; stop_graph_server()

## ③ 모델·엔드포인트·시스템 프롬프트 바꾸기

두뇌는 **langgraph 그래프 하나**(deepagents 가 생성). 별도 어댑터 클래스는 없습니다. 대부분 환경변수만 바꿔도 충분하지만, 셀에서 명시적으로 주고 싶을 땐:

```python
from jlab_sidebar_chatbot import start_graph_server

# 사내 vLLM + 커스텀 프롬프트 한 번에
start_graph_server(
    api_key="<사내 키>",
    base_url="https://<사내 vllm host>/v1",   # 실 OpenAI 면 비우기
    model="<served-model-name>",              # 실 OpenAI 면 "gpt-4o-mini" 등
    system_prompt="너는 SQL 전문가야. 쿼리 위주로 답해.",
)
```

직접 그래프를 만들어 도구를 추가하고 싶으면:

```python
from jlab_sidebar_chatbot import build_chat_graph, start_graph_server

graph = build_chat_graph(
    api_key="<키>", base_url="https://<vllm>/v1", model="<name>",
    tools=[my_tool_fn, ...],
)
start_graph_server(graph=graph)
```

- 멀티턴은 langgraph 체크포인터(`InMemorySaver`) + `thread_id`(=세션) 가 관리합니다.
- 인자가 환경변수보다 우선합니다. 키를 셀에 적기 싫으면 `getpass.getpass()` 로 받으세요.

## ④ 정리 — 설치한 익스텐션 삭제

데모가 끝나면 아래 셀로 두뇌 서버를 멈추고 패키지(프론트 labextension 포함)를 제거합니다. **jupyter 재시작 없이** 브라우저 새로고침만 하면 우측 💬 탭이 사라집니다.

In [ ]:
# (정리) 데모가 끝나면 두뇌 서버를 멈추고, 설치했던 익스텐션 패키지를 제거합니다.
from jlab_sidebar_chatbot import stop_graph_server
stop_graph_server()                       # 8765 두뇌 서버 중지 (먼저 — uninstall 전에)

%pip uninstall -y jlab_sidebar_chatbot    # 프론트 labextension 자산 포함 패키지 삭제
# → 브라우저를 '새로고침'하면 우측 💬 탭이 사라집니다 (jupyter 서버 재시작 불필요)